# Actividad — SVR para consumo energético de edificios
**Aprendizaje Supervisado · Sesión 6 · Regresión con máquinas de vectores de soporte**

**Qué haremos.** En la presentación vimos que SVR controla el error con un **tubo $\varepsilon$-insensible**
en vez de penalizar todos los residuos, y que el **truco del kernel** (lineal, polinomial, RBF) le permite
curvarse sin construir columnas nuevas a mano. Hoy aplicas todo eso a un problema real: predecir el
**consumo eléctrico diario (kWh)** de un edificio a partir del clima y sus características, donde la
relación con la temperatura **no es una línea recta** sino una curva en forma de U.

Trabaja **de forma individual**. Las celdas con `...` están incompletas: reemplaza los puntos suspensivos
con el código que falta (el comentario `✏️` te da una pista de qué se espera).

## Parte 0 — Los datos

Cada fila es un edificio-día con 5 variables de entrada: `temperatura_c`, `superficie_m2`, `ocupantes`,
`aislamiento` (0 malo .. 1 excelente) y `fin_semana` (0/1). La salida es `consumo_kwh`. El consumo depende
de forma **no lineal** de la temperatura (sube con frío y con calor, mínimo cerca de los 21°C), y esa
curvatura se vuelve más suave cuanto mejor aislado está el edificio.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

datos = pd.read_csv("consumo_energetico.csv")
FEATS = ["temperatura_c", "superficie_m2", "ocupantes", "aislamiento", "fin_semana"]
X = datos[FEATS].values
y = datos["consumo_kwh"].values

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42)
kf = KFold(5, shuffle=True, random_state=0)

def rmse(m, X, y):
    return np.sqrt(mean_squared_error(y, m.predict(X)))

print(f"n entrenamiento = {len(Xtr)}   n prueba = {len(Xte)}   variables = {FEATS}")
datos.describe().round(2).T

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
sca = ax.scatter(datos["temperatura_c"], datos["consumo_kwh"], c=datos["aislamiento"], cmap="viridis", s=14, alpha=0.6)
fig.colorbar(sca, ax=ax, label="aislamiento")
ax.set_xlabel("temperatura exterior (°C)"); ax.set_ylabel("consumo (kWh/día)")
ax.set_title("Consumo energético vs. temperatura: una curva en U");

✏️ **Pregunta 0.1.** Mirando la gráfica: ¿por qué un modelo lineal (una sola recta) no podría ajustar bien esta relación? ¿En qué rango de temperatura el consumo es más bajo?

**Respuesta:**

## Parte 1 — SVR lineal como punto de partida  *(presentación A4, C1)*

Ajustamos un SVR con `kernel="linear"`. El escalador va dentro del `Pipeline` porque SVR usa distancias/productos
punto, y son sensibles a la escala de cada variable.

In [ ]:
lineal = ...     # ✏️ make_pipeline(StandardScaler(), SVR(kernel="linear", C=10, epsilon=1.0)).fit(Xtr, ytr)
print(f"SVR lineal   RMSE entrenamiento = {rmse(lineal, Xtr, ytr):.2f}    RMSE prueba = {rmse(lineal, Xte, yte):.2f}")
print(f"Vectores de soporte: {lineal[-1].support_.shape[0]} de {len(Xtr)}")

✏️ **Pregunta 1.1.** El RMSE del SVR lineal, ¿es alto o bajo en comparación con la escala de `consumo_kwh` (revisa la tabla de la Parte 0)? ¿Por qué un kernel lineal tiene este problema aquí, si recordamos la Parte 0?

**Respuesta:**

## Parte 2 — SVR con kernel RBF  *(presentación C1–C2)*

Repetimos con `kernel="rbf"` y los hiperparámetros **por defecto** de scikit-learn (sin ajustar nada todavía).

In [ ]:
rbf_defecto = ...     # ✏️ make_pipeline(StandardScaler(), SVR(kernel="rbf")).fit(Xtr, ytr)
print(f"SVR RBF (por defecto)  RMSE entrenamiento = {rmse(rbf_defecto, Xtr, ytr):.2f}    RMSE prueba = {rmse(rbf_defecto, Xte, yte):.2f}")
print(f"Hiperparámetros usados: C={rbf_defecto[-1].C}, epsilon={rbf_defecto[-1].epsilon}, gamma={rbf_defecto[-1].gamma}")
print(f"Vectores de soporte: {rbf_defecto[-1].support_.shape[0]} de {len(Xtr)}")

✏️ **Pregunta 2.1.** Compara el RMSE de prueba del RBF por defecto contra el del SVR lineal de la Parte 1. ¿Confirma esto la idea de la presentación sobre por qué el kernel lineal falla aquí?

**Respuesta:**

## Parte 3 — Efecto de $C$ y de $\varepsilon$  *(presentación B4–B6)*

Primero variamos $C$ dejando $\varepsilon=1.0$ fijo; luego variamos $\varepsilon$ dejando $C=50$ fijo.
En ambos casos usamos kernel RBF con `gamma="scale"`.

In [ ]:
valores_C = [0.01, 0.1, 1, 10, 100, 1000]
filas_c = []
for C in valores_C:
    m = ...     # ✏️ make_pipeline(StandardScaler(), SVR(kernel="rbf", C=C, epsilon=1.0)).fit(Xtr, ytr)
    filas_c.append({"C": C, "rmse_train": rmse(m, Xtr, ytr), "rmse_test": rmse(m, Xte, yte),
                     "vectores_soporte": int(m[-1].support_.shape[0])})
tabla_c = pd.DataFrame(filas_c)
print(tabla_c.round(2).to_string(index=False))

fig, ax1 = plt.subplots(figsize=(6.3, 3.6))
ax1.plot(tabla_c["C"], tabla_c["rmse_test"], "o-", color="#065A82", label="RMSE prueba")
ax1.set_xscale("log"); ax1.set_xlabel("C"); ax1.set_ylabel("RMSE prueba", color="#065A82")
ax2 = ax1.twinx()
ax2.plot(tabla_c["C"], tabla_c["vectores_soporte"], "s--", color="#E07A1F", label="# vect. soporte")
ax2.set_ylabel("# vectores de soporte", color="#E07A1F")
ax1.set_title("Efecto de C sobre el error y el número de vectores de soporte");

In [ ]:
valores_eps = [0.1, 0.5, 1, 2, 4, 8]
filas_e = []
for eps in valores_eps:
    m = ...     # ✏️ make_pipeline(StandardScaler(), SVR(kernel="rbf", C=50, epsilon=eps)).fit(Xtr, ytr)
    filas_e.append({"epsilon": eps, "rmse_train": rmse(m, Xtr, ytr), "rmse_test": rmse(m, Xte, yte),
                     "vectores_soporte": int(m[-1].support_.shape[0])})
tabla_e = pd.DataFrame(filas_e)
print(tabla_e.round(2).to_string(index=False))

✏️ **Pregunta 3.1.** (a) ¿Qué le pasa al RMSE de prueba con $C$ muy chica? ¿Y con $C$ muy grande, el RMSE de prueba sigue mejorando o empieza a empeorar? (b) ¿Cómo cambia el número de vectores de soporte al subir $C$? (c) ¿Cómo cambia el número de vectores de soporte al subir $\varepsilon$, y por qué tiene sentido con lo que viste en la presentación (parte B)?

**Respuesta:**

## Parte 4 — Comparación de kernels: lineal, polinomial y RBF  *(presentación C3–C4)*

Ahora comparamos los tres kernels **con el mismo protocolo**: mismo $C$, mismo $\varepsilon$, mismos
conjuntos de entrenamiento y prueba.

In [ ]:
modelos4 = {
    "lineal": SVR(kernel="linear", C=50, epsilon=1.0),
    "poly(3)": ...,     # ✏️ SVR(kernel="poly", degree=3, coef0=1, C=50, epsilon=1.0)
    "rbf": SVR(kernel="rbf", C=50, epsilon=1.0),
}
filas4 = []
for nombre, svr in modelos4.items():
    m = make_pipeline(StandardScaler(), svr).fit(Xtr, ytr)
    filas4.append({"kernel": nombre, "rmse_train": rmse(m, Xtr, ytr), "rmse_test": rmse(m, Xte, yte),
                    "vectores_soporte": int(m[-1].support_.shape[0])})
tabla4 = pd.DataFrame(filas4)
print(tabla4.round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.bar(tabla4["kernel"], tabla4["rmse_test"], color=["#607D8B", "#1C7293", "#02C39A"])
ax.set_ylabel("RMSE de prueba"); ax.set_title("Comparación de kernels (mismo C, mismo ε)");

✏️ **Pregunta 4.1.** (a) ¿Qué kernel tiene el peor RMSE de prueba y por qué (relaciónalo con la Parte 0)? (b) ¿Qué tan parecidos son poly(3) y RBF aquí? (c) ¿Cuál de los dos usa menos vectores de soporte?

**Respuesta:**

## Parte 5 — Ajuste de $C$, $\varepsilon$ y $\gamma$ con `GridSearchCV`  *(presentación D1–D2)*

Buscamos los tres hiperparámetros del kernel RBF **juntos**, con validación cruzada de 5 particiones.
La rejilla es moderada para que la celda no tarde demasiado; en un proyecto real valdría la pena
una rejilla más fina alrededor del óptimo encontrado aquí.

In [ ]:
pipe_rbf = make_pipeline(StandardScaler(), SVR(kernel="rbf"))
malla = {"svr__C": np.logspace(0, 3, 6), "svr__gamma": np.logspace(-2, 1, 6), "svr__epsilon": [0.5, 1.0, 2.0]}
bus = ...     # ✏️ GridSearchCV(pipe_rbf, malla, cv=kf, scoring="neg_root_mean_squared_error", n_jobs=-1)
bus.fit(Xtr, ytr)
print("Mejores hiperparámetros:", bus.best_params_)
print(f"RMSE de validación = {-bus.best_score_:.2f}    RMSE de prueba = {rmse(bus, Xte, yte):.2f}")
print(f"Vectores de soporte: {bus.best_estimator_[-1].support_.shape[0]} de {len(Xtr)}")

✏️ **Pregunta 5.1.** (a) ¿Qué combinación de $C$, $\gamma$ y $\varepsilon$ eligió `GridSearchCV`? (b) ¿Mejora el RMSE de prueba frente al RBF con hiperparámetros por defecto de la Parte 2? (c) Si tuvieras más tiempo de cómputo, ¿cómo ajustarías la rejilla para refinar la búsqueda alrededor de lo encontrado?

**Respuesta:**

## Parte 6 — Comparación final contra los modelos de sesiones anteriores  *(presentación E2)*

Ponemos el mejor SVR (de la Parte 5) a competir contra **OLS**, **Ridge** y **Lasso** (con validación
cruzada para su hiperparámetro), todos evaluados en el **mismo** conjunto de prueba.

In [ ]:
modelos6 = {
    "OLS": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 50), cv=kf)),
    "Lasso": ...,     # ✏️ make_pipeline(StandardScaler(), LassoCV(alphas=np.logspace(-3, 2, 60), cv=kf, max_iter=100000))
    "SVR lineal": make_pipeline(StandardScaler(), SVR(kernel="linear", C=50, epsilon=1.0)),
    "SVR RBF (GridSearchCV)": bus.best_estimator_,
}
filas6 = []
for nombre, m in modelos6.items():
    if nombre != "SVR RBF (GridSearchCV)":
        m.fit(Xtr, ytr)
    filas6.append({"modelo": nombre, "rmse_prueba": round(rmse(m, Xte, yte), 3)})
comparacion = pd.DataFrame(filas6).sort_values("rmse_prueba").reset_index(drop=True)
print(comparacion.to_string(index=False))

entregable = comparacion.copy()
entregable["comentario"] = "✏️ escribe tu comentario"
entregable.to_csv("entregable_unidad6.csv", index=False)
entregable

✏️ **Pregunta 6.1.** (a) Ordena los modelos de mejor a peor en prueba. ¿Cuánto gana el mejor SVR sobre el mejor modelo lineal (OLS/Ridge/Lasso)? (b) ¿Por qué Ridge y Lasso no logran capturar la curva en U, aunque regularicen bien? (c) Esta comparación viene de **un solo** reparto entrenamiento/prueba: ¿qué harías para confiar más en la diferencia entre SVR y los modelos lineales?

**Respuesta:**

## Parte 7 — Interpretación: ¿qué aprendió el modelo sobre la temperatura?

Fijamos todas las variables en su **mediana** y vamos moviendo solo `temperatura_c`, para ver qué
predice cada modelo a lo largo del rango de temperaturas. Esto se llama una curva de **dependencia parcial**
simple.

In [ ]:
mediana = datos[FEATS].median()
rejilla_temp = np.linspace(-5, 40, 25)
filas7 = []
for t in rejilla_temp:
    fila = mediana.copy()
    fila["temperatura_c"] = t
    xf = ...     # ✏️ fila[FEATS].values.reshape(1, -1)
    filas7.append({"temperatura": t,
                    "OLS": modelos6["OLS"].predict(xf)[0],
                    "SVR_RBF": modelos6["SVR RBF (GridSearchCV)"].predict(xf)[0]})
dep = pd.DataFrame(filas7)

fig, ax = plt.subplots(figsize=(6.3, 3.8))
ax.plot(dep["temperatura"], dep["OLS"], color="#607D8B", lw=2, label="OLS")
ax.plot(dep["temperatura"], dep["SVR_RBF"], color="#02C39A", lw=2.3, label="SVR RBF (GridSearchCV)")
ax.set_xlabel("temperatura (°C), el resto de las variables fijas en su mediana")
ax.set_ylabel("consumo predicho (kWh/día)")
ax.set_title("Dependencia parcial de la temperatura")
ax.legend();

✏️ **Pregunta 7.1.** (a) ¿La curva de OLS es capaz de mostrar un mínimo de confort? (b) ¿La curva de SVR RBF sí lo muestra? ¿Alrededor de qué temperatura? (c) ¿Por qué esta diferencia importa para una recomendación práctica a un gestor de edificios (por ejemplo, para decidir cuándo encender calefacción o aire acondicionado)?

**Respuesta:**

## Parte 8 — Recomendación final

✏️ **Pregunta 8.1.** Redacta una recomendación breve (6 a 8 líneas) para un colega que va a modelar el
consumo energético de otro conjunto de edificios: qué kernel usarías y por qué, qué pasos seguirías
(escalado, validación cruzada, elección de $C$/$\varepsilon$/$\gamma$), qué mejora de RMSE esperarías
frente a un modelo lineal **en datos como estos**, qué limitaciones tiene SVR (costo computacional,
interpretabilidad frente a un modelo lineal) y qué cuidados tomaste para no engañarte (conjunto de
prueba, escalado dentro del pipeline).

**Respuesta:**

---
**Reto opcional — ¿importa estandarizar en SVR?** Comparamos el kernel RBF **con** y **sin** `StandardScaler`,
manteniendo los mismos $C$ y $\varepsilon$. Recuerda: RBF depende de distancias $\lVert x - x'\rVert$, y
nuestras variables tienen escalas muy distintas (`temperatura_c` en decenas, `aislamiento` entre 0 y 1).

In [ ]:
con_escalado = make_pipeline(StandardScaler(), SVR(kernel="rbf", C=50, epsilon=1.0)).fit(Xtr, ytr)
sin_escalado = SVR(kernel="rbf", C=50, epsilon=1.0).fit(Xtr, ytr)
for nombre, m in [("con StandardScaler", con_escalado), ("sin escalar", sin_escalado)]:
    nsv = m[-1].support_.shape[0] if hasattr(m, "steps") else m.support_.shape[0]
    print(f"{nombre:22s} RMSE prueba = {rmse(m, Xte, yte):.2f}   vectores de soporte = {nsv}")

✏️ **Pregunta del reto.** ¿Qué diferencia hay en el RMSE de prueba con y sin escalar? ¿Qué variable crees que termina dominando la distancia cuando no se escala, y por qué?

**Respuesta:**